# Recurso Hídrico — Plan de Ordenamiento del Recurso Hídrico (PORH)

> **Contexto de dominio:** [`docs/fuentes/recurso_hidrico.md`](../../docs/fuentes/recurso_hidrico.md)
> **Bloque:** A — Gestión | **Línea:** `recurso_hidrico`
> **Variables principales:** OD (mg/L), DBO₅ (mg/L), DQO (mg/L), pH, SST (mg/L), caudal (m³/s)
> **Modelos sugeridos:** SARIMA, SARIMAX, XGBOOST, RANDOM_FOREST
> **Flujo:** `Plan.md` sección 3 — ciclo estadístico completo.

---

## ¿Qué es el PORH y por qué importa en Colombia?

El **Plan de Ordenamiento del Recurso Hídrico (PORH)** es el principal instrumento de planificación estratégica para la gestión del agua en Colombia. Su propósito es fijar la destinación, clasificación y usos de los cuerpos de agua continentales superficiales, estableciendo las normas, condiciones y programas de seguimiento necesarios para mantener y recuperar la calidad del recurso en un horizonte mínimo de **diez años**.

El PORH se articula dentro del Sistema Nacional Ambiental (SINA) y se apoya en modelos de simulación de calidad del agua, índices de alteración potencial y análisis geoespaciales que permiten cuantificar las cargas contaminantes y determinar la **capacidad asimilativa** de las fuentes hídricas. En la práctica, es el instrumento que responde: *¿puede este río seguir recibiendo vertimientos sin colapsar ecosistémicamente?*

## Instituciones responsables

| Institución | Rol |
|---|---|
| **MADS** | Rector de política; emite Res. 0751/2018 (guía técnica PORH) |
| **IDEAM** | Lineamientos de modelación, redes de monitoreo, Estudio Nacional del Agua |
| **CARs** | Formulan y hacen seguimiento a los PORH (CAR, CORANTIOQUIA, CORTOLIMA) |
| **Municipios / ESP** | Principales generadores de vertimientos; ejecutan PSMV |

## Variables de calidad del agua analizadas

| Variable | Unidad | Umbral crítico | Interpretación |
|---|---|---|---|
| **OD** (Oxígeno Disuelto) | mg/L | < 4 mg/L | Anoxia: riesgo para peces y vida acuática |
| **DBO₅** | mg/L O₂ | > 30 mg/L (Res. 631/2015) | Alta carga orgánica: vertimientos domésticos |
| **DQO** | mg/L O₂ | > 200 mg/L | Contaminación química industrial |
| **pH** | Unidades | < 5 o > 9 | Acidez/alcalinidad extrema |
| **SST** | mg/L | > 100 mg/L (típico) | Turbiedad, erosión de cuencas |
| **Coliformes fecales** | NMP/100 mL | > 1000 (uso recreativo) | Contaminación microbiológica |

## Preguntas que responde este análisis

1. ¿Cuál es la tendencia histórica del OD y la DBO₅? ¿Hay mejoría o deterioro de la calidad del agua?
2. ¿Se exceden los límites de la Resolución 2115/2007 (agua potable) o Res. 631/2015 (vertimientos)?
3. ¿Qué períodos del año concentran el mayor riesgo de excedencias normativas (estiaje vs. lluvia)?
4. ¿El ICA (Índice de Calidad del Agua) ubica el tramo en categoría aceptable, regular o deficiente?
5. ¿Qué modelo predice mejor el OD a futuro para apoyar la toma de decisiones del PORH?

---

**Antes de ejecutar:** Leer `docs/fuentes/recurso_hidrico.md` para conocer los índices IACAL, ICA e ICO, la normativa aplicable y las fuentes de datos del IDEAM/CARs.

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.features.climate import load_oni, enso_lagged
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "recurso_hidrico"
VARIABLE = "od"
UNIDAD = "mg/L"

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio

Esta celda carga la ficha técnica de la línea `recurso_hidrico` directamente desde `docs/fuentes/`. La ficha contiene el resumen ejecutivo del PORH, las variables de calidad con sus rangos típicos, la normativa y los índices de evaluación estandarizados por el IDEAM y el MADS.

Se recomienda revisar especialmente:
- La tabla de **Variables ambientales**: OD (> 4 mg/L), DBO₅, DQO, SST, pH (5-9), coliformes fecales.
- Los **índices de evaluación**: IACAL (presión por contaminación), ICA (calidad agregada), ICO (contaminación orgánica ICOMO y sólidos ICOSUS), BMWP-Col (bioindicador macroinvertebrados).
- Las **fuentes de datos**: resultados de campañas fisicoquímicas y microbiológicas de CARs, IDEAM-DHIME, RURH.

> **Nota sobre outliers (ADR-002):** Los valores extremos de DBO₅ o DQO suelen indicar eventos de vertimiento puntual o lluvias que arrastran cargas difusas. Son señal ambiental real. No eliminar sin análisis previo. El módulo `preprocessing/outliers.py` es opt-in explícito.

> **Nota sobre múltiples variables:** En esta línea se trabaja con varias variables simultáneamente (OD, DBO₅, DQO, pH, SST). En producción, repetir la secuencia de análisis para cada variable o usar `for variable in ["od", "dbo5", "dqo", "ph", "sst"]:`.

In [ ]:
ficha = DOCS_FUENTES / "recurso_hidrico.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos

### ¿Qué datos necesito?

Para el análisis del recurso hídrico (PORH) se requiere una serie temporal de calidad del agua con las siguientes columnas:

| Columna | Tipo | Descripción |
|---|---|---|
| `fecha` | datetime | Fecha de la campaña de monitoreo |
| `od` | float | Oxígeno Disuelto (mg/L) — variable primaria |
| `dbo5` | float (opcional) | Demanda Bioquímica de Oxígeno a 5 días (mg/L) |
| `dqo` | float (opcional) | Demanda Química de Oxígeno (mg/L) |
| `ph` | float (opcional) | pH del agua (unidades) |
| `sst` | float (opcional) | Sólidos Suspendidos Totales (mg/L) |
| `caudal` | float (opcional) | Caudal (m³/s) — para calcular cargas |
| `estacion` | str (opcional) | Código o nombre del punto de monitoreo |

### Fuentes oficiales de datos en Colombia

- **IDEAM — DHIME:** [dhime.ideam.gov.co](http://dhime.ideam.gov.co) — Series fisicoquímicas de calidad de agua.
- **IDEAM — SIRH:** [sirh.ideam.gov.co](http://sirh.ideam.gov.co) — Registros históricos de calidad y usuarios.
- **CARs regionales:** Datos de campañas de monitoreo periódico (laboratorio acreditado). Ej. CAR Cundinamarca, CORANTIOQUIA.
- **Laboratorios acreditados:** Resultados fisicoquímicos y microbiológicos bajo protocolos IDEAM.

### Frecuencia y horizonte recomendados

- **Frecuencia mínima:** Mensual (por campaña). Puede ser trimestral en cuencas con menor presión.
- **Horizonte mínimo:** 5 años para análisis de tendencia. El PORH tiene horizonte de 10 años.

### Consideración sobre períodos hidrológicos

En Colombia, la calidad del agua varía significativamente entre períodos **hidrológico alto** (lluvia) e **hidrológico bajo** (estiaje). Es recomendable estratificar el análisis por régimen hidrológico. La variable `caudal` permite hacer esta distinción: los meses de mayor DBO₅ con caudal bajo indican menor dilución y mayor riesgo.

> **Nota:** El dataset sintético de ejemplo solo incluye la variable `od`. Para datos reales, incluir todas las variables relevantes y aplicar el bucle de análisis por variable.

In [ ]:
# df = load_csv("data/raw/recurso_hidrico.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "od": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

## 2. Validación y EDA

### ¿Qué hace `validate()` en esta línea?

La función `validate()` aplica rangos físicos específicos para la línea `recurso_hidrico` definidos en `config.py`. Verifica que:

- El OD esté en rango físico posible (0 a 14 mg/L a temperatura ambiente).
- La DBO₅ y DQO sean valores positivos.
- El pH esté entre 0 y 14, con alerta si < 5 o > 9.
- Los SST sean positivos y dentro de rangos típicos (< 1000 mg/L para cuerpos no altamente turbios).
- No haya fechas duplicadas ni saltos temporales anómalos.

### Señales de alerta en la validación

Si `val.summary()` reporta problemas:

- **OD = 0 mg/L:** Zona anóxica confirmada. Señal crítica de impacto por vertimientos. No es error.
- **DBO₅ > 200 mg/L:** Posible vertimiento industrial o doméstico sin tratamiento. Verificar fuente puntual.
- **pH fuera de 5-9:** Acidez o alcalinidad extrema. Verificar contra minería o vertimientos industriales.
- **Periodos sin datos:** Los monitoreos de calidad son costosos y discontinuos. La sección 6 aplica imputación.

> **ADR-006:** Usar siempre `validate(df, linea_tematica="recurso_hidrico")` para rangos específicos de calidad del agua. Los valores extremos de DBO₅ o SST en períodos de lluvia son señal de arrastre de cargas difusas, no errores de medición.

> **ADR-002:** En calidad del agua, los outliers son frecuentemente eventos reales de vertimiento o arrastre de sedimentos. No aplicar clipping. Documentar y analizar su origen.

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_recurso_hidrico.html",
       title="EDA — Recurso Hídrico", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria

### ¿Qué buscar en las gráficas?

Las visualizaciones de esta sección permiten leer el comportamiento de la calidad del agua antes de aplicar estadística formal. En el contexto del PORH, se deben identificar:

**En la serie temporal del OD (plot_series):**
- Tendencia: ¿el OD tiende a disminuir en los últimos años? Indicaría deterioro progresivo de la calidad del agua, posiblemente por aumento de cargas orgánicas (crecimiento urbano, agroindustria).
- Estacionalidad: el OD es inversamente proporcional a la temperatura del agua. En aguas más frías (época seca en zonas altas) el OD sube; en aguas cálidas baja. Verificar si la tendencia real está enmascarada por el ciclo térmico estacional.
- Eventos de anoxia: caídas bruscas del OD por debajo de 4 mg/L son señal de incidente de contaminación o evento de lluvia que arrastra cargas difusas.

**En las medias estacionales (plot_seasonal_means):**
- Comparar el OD mensual promedio contra el umbral crítico de 4 mg/L. Los meses consistentemente por debajo de este umbral son zonas de riesgo permanente para la vida acuática.
- Si la DBO₅ muestra picos en meses secos: indica que el río pierde capacidad de dilución y la carga orgánica se concentra — escenario de alta presión sobre la capacidad asimilativa.
- Para el SST: los picos típicamente coinciden con el inicio de la temporada de lluvia (primer lavado de cuencas) en marzo-abril y octubre-noviembre.

In [ ]:
plot_series(df, "fecha", "od", title="Recurso Hídrico — od (mg/L)")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "od", period="month")
plt.show()

## 3c. Capacidad de asimilacion y modelo de calidad QUAL2K

La **capacidad de asimilacion** es la cantidad de carga contaminante que un rio puede recibir sin superar los objetivos de calidad del PORH (Res. 0751/2018).

El modelo **Streeter-Phelps** (base de QUAL2K/QUAL2Kw) describe el deficit de OD:

```
DBO(x) = DBO0 * exp(-Kd * x/U)            Kd = tasa descomposicion (1/dia)
OD_deficit(x) = Dc * exp(-Ka * x/U)       Ka = tasa reaireacion (1/dia)
               - (Kd*DBO0)/(Ka-Kd) * [exp(-Kd*x/U) - exp(-Ka*x/U)]
Donde U = velocidad media del rio (m/s), x = distancia aguas abajo (km)
```

**QUAL2K** extiende este modelo con temperatura, nitrificacion, algas y sedimentos. Implementacion Python: `qual2kpy` o manual con ODE de scipy.

> **BMWP-Col (macroinvertebrados):** bioindicador biologico obligatorio en el diagnostico del PORH, complementa el ICA fisicoquimico con informacion de la calidad ecologica del agua.

In [ ]:
# Modelo Streeter-Phelps simplificado — perfil DBO y deficit de OD aguas abajo
# Simula la capacidad de asimilacion de un rio receptor de vertimiento

# Parametros del cuerpo de agua (rio andino tipico)
Q_rio = 5.0       # m3/s caudal receptor
U_vel = 0.4       # m/s velocidad media
OD_sat = 8.5      # mg/L saturacion OD (a ~1500 msnm)
OD_ini = 6.2      # mg/L OD aguas arriba del vertimiento
Kd = 0.25         # 1/dia tasa descomposicion DBO
Ka = 0.80         # 1/dia tasa reaireacion (K2)

# Vertimiento (punto de descarga PTAR municipal)
Q_vert = 0.15     # m3/s caudal vertido
DBO_vert = 180    # mg/L DBO5 del vertimiento
OD_vert = 1.0     # mg/L OD del vertimiento

# Mezcla inmediata aguas abajo del punto de descarga
DBO_mix = (Q_rio * 2.0 + Q_vert * DBO_vert) / (Q_rio + Q_vert)  # 2.0 mg/L DBO rio
OD_mix  = (Q_rio * OD_ini + Q_vert * OD_vert) / (Q_rio + Q_vert)
Dc_ini  = OD_sat - OD_mix  # deficit inicial

# Perfil a lo largo del rio (0 a 80 km)
x_km = np.linspace(0, 80, 200)
t = x_km * 1000 / (U_vel * 86400)  # dias de viaje

DBO_x = DBO_mix * np.exp(-Kd * t)
def_od = (Dc_ini * np.exp(-Ka * t)
          - (Kd * DBO_mix) / (Ka - Kd) * (np.exp(-Kd * t) - np.exp(-Ka * t)))
OD_x = np.clip(OD_sat - def_od, 0, OD_sat)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(x_km, DBO_x, lw=2, color='#e74c3c', label='DBO5 (mg/L)')
ax1.axhline(5, color='orange', ls='--', lw=1.5, label='Objetivo calidad PORH DBO=5')
x_meta_dbo = x_km[DBO_x < 5][0] if (DBO_x < 5).any() else 80
ax1.axvline(x_meta_dbo, color='gray', ls=':', lw=1)
ax1.set_xlabel('Distancia aguas abajo (km)'); ax1.set_ylabel('DBO5 (mg/L)')
ax1.set_title('Perfil DBO — Streeter-Phelps (base QUAL2K)', fontweight='bold')
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

ax2.plot(x_km, OD_x, lw=2, color='#2980b9', label='OD (mg/L)')
ax2.axhline(4.0, color='red', ls='--', lw=1.5, label='Minimo OD uso piscicola=4 mg/L')
ax2.axhline(OD_sat, color='green', ls=':', lw=1, label=f'OD saturacion={OD_sat}')
od_min_km = x_km[OD_x.argmin()]
ax2.axvline(od_min_km, color='gray', ls=':', lw=1, label=f'Minimo OD en {od_min_km:.1f} km')
ax2.set_xlabel('Distancia aguas abajo (km)'); ax2.set_ylabel('OD (mg/L)')
ax2.set_title('Deficit de OD — Curva de sagging (capacidad asimilacion)', fontweight='bold')
ax2.legend(fontsize=7); ax2.grid(alpha=0.3)

plt.suptitle('Modelo QUAL2K simplificado — PORH (Res. 0751/2018 MADS)',
             fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show()

od_min = OD_x.min()
print(f'OD minimo: {od_min:.2f} mg/L a {od_min_km:.1f} km del vertimiento')
print(f'DBO5 cumple objetivo (< 5 mg/L) a partir de: {x_meta_dbo:.1f} km')
print(f'Capacidad asimilacion: Q_rio={Q_rio} m3/s puede asimilar ~{Q_vert*DBO_vert:.0f} g/s DBO')
print('BMWP-Col recomendado en km 0, 10, 30 y 60 para validar el modelo')

## 3b. Covariable ENSO (ONI) — Por qué y con qué lag

### ¿Por qué incorporar ENSO en el análisis de calidad del agua?

El fenómeno ENSO afecta la calidad del agua a través de su impacto sobre el caudal y la temperatura. La relación es indirecta pero consistente:

- **El Niño (ONI > 0.5):** Reduce los caudales. Menor dilución de vertimientos → mayor concentración de DBO₅, DQO y coliformes. Temperatura del agua más alta → menor OD por saturación. Escenario de estrés máximo para la calidad del agua: mínima oferta, máxima concentración de contaminantes.
- **La Niña (ONI < -0.5):** Aumenta los caudales. Mayor dilución → OD más alto, DBO₅ más baja en concentración (pero la carga total puede ser mayor). Los SST aumentan por arrastre de sedimentos desde cuencas sin cobertura.

### ¿Con qué lag afecta ENSO a la calidad del agua?

Para la línea `recurso_hidrico`, el impacto del ENSO sobre la calidad del agua superficial ocurre principalmente a través del caudal, que responde con un lag menor que los acuíferos. El lag configurado en `config.ENSO_LAG_MESES["recurso_hidrico"]` refleja el desfase entre el pico del ONI y la respuesta hidrológica en cuerpos de agua superficiales.

> **ADR-007:** Usar siempre `enso_lagged()` con el lag específico de la línea. Para calidad de agua superficial, el lag es típicamente menor que para acuíferos porque la respuesta del caudal superficial es más rápida. El valor exacto se obtiene de `config.ENSO_LAG_MESES`.

### Uso analítico de las columnas ENSO

Las columnas `oni_lag` y `enso_fase` generadas son covariables para los modelos SARIMAX y XGBoost. Permiten que el modelo capte que la relación OD-temperatura-caudal-ENSO es no estacionaria y depende de la fase climática en curso. Incluirlas mejora la capacidad predictiva en años de variabilidad climática extrema.

In [ ]:
# --- Covariable ENSO (lag específico para Recurso Hídrico) ---
oni = load_oni()  # Descarga ONI desde NOAA
df = enso_lagged(df, oni, date_col="fecha", linea_tematica="recurso_hidrico")
print("Columnas ENSO agregadas:", [c for c in df.columns if "oni" in c or "enso" in c])

## 4. Estadística descriptiva

In [ ]:
summarize(df)

## 5. Inferencial — Estacionariedad y tendencia

### ¿Por qué verificar estacionariedad? (ADR-004)

Las series de calidad del agua (OD, DBO₅, pH) frecuentemente presentan tendencias no estacionarias asociadas al crecimiento de las presiones antrópicas (expansión urbana, vertimientos industriales). Antes de modelar, es obligatorio verificar la estacionariedad con las pruebas ADF y KPSS.

| Prueba | H₀ | Acción si se rechaza |
|---|---|---|
| **ADF** | Serie no estacionaria (raíz unitaria) | Serie es estacionaria → ARIMA con d=0 |
| **KPSS** | Serie es estacionaria | Serie no estacionaria → diferenciar (d=1) |

**Señales de alerta específicas para calidad del agua:**
- Si el OD muestra tendencia **decreciente** significativa: evidencia de deterioro progresivo de la calidad del agua. Reportar inmediatamente en el PORH.
- Si la DBO₅ muestra tendencia **creciente**: aumento de cargas orgánicas vertidas; señal de que los PSMV (Planes de Saneamiento y Manejo de Vertimientos) no están siendo efectivos.
- Si el pH muestra raíz unitaria: el proceso de acidificación/alcalinización es persistente; posiblemente relacionado con actividad minera sostenida.

### Interpretación de Mann-Kendall para calidad del agua

- `trend = "decreasing"` en OD con p < 0.05: deterioro progresivo estadísticamente significativo. Es el hallazgo más crítico del PORH.
- `slope` negativo en OD: pérdida de mg/L por año. Proyectar cuántos años faltan para cruzar el umbral de 4 mg/L si la tendencia continúa.
- `trend = "increasing"` en DBO₅: aumento de carga orgánica con el tiempo. Calcular cuándo se alcanzará el límite de la Res. 631/2015.

> **ADR-004:** Obligatorio ADF + KPSS antes de cualquier modelo ARIMA. Para variables de calidad como DBO₅ que presentan ceros o valores muy bajos en estiaje, considerar transformación logarítmica antes de la prueba.

In [ ]:
ts = df.set_index("fecha")["od"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5b. Análisis de excedencias normativas — Resoluciones 2115/2007 y 631/2015

### ¿Qué normas aplican al recurso hídrico?

Colombia cuenta con dos normas principales para la calidad del agua, con umbrales diferenciados según el uso del recurso:

**Resolución 2115 de 2007 — Agua potable (NORMA_AGUA_POTABLE en config.py):**

| Parámetro | Valor máximo permisible | Uso |
|---|---|---|
| pH | 6.5 – 9.0 | Consumo humano |
| OD | > 4 mg/L (indirecto) | No especificado pero es criterio de calidad |
| Conductividad | < 1000 µS/cm | Consumo humano |
| Coliformes totales | 0 NMP/100 mL | Consumo humano directo |

**Resolución 631 de 2015 — Vertimientos (NORMA_VERTIMIENTOS en config.py):**

| Parámetro | Límite máximo permisible | Fuente |
|---|---|---|
| DBO₅ (doméstico) | 90 mg/L | Res. 631/2015 |
| DQO (doméstico) | 180 mg/L | Res. 631/2015 |
| SST (doméstico) | 90 mg/L | Res. 631/2015 |
| pH | 6.0 – 9.0 | Res. 631/2015 |

### Interpretación del reporte de excedencias para OD

Para el OD, el criterio crítico es **4 mg/L** (por debajo = anoxia, riesgo para vida acuática). El reporte mostrará:
- Porcentaje de meses con OD < 4 mg/L.
- Distribución temporal de las excedencias — ¿se concentran en meses secos?
- Comparación contra las guías OMS si están disponibles en `config.NORMA_OMS`.

### ICA — Índice de Calidad del Agua (config.ICA_CATEGORIES)

El ICA agrega múltiples variables de calidad en un índice escalar entre 0 y 1:

| ICA | Categoría | Descripción |
|---|---|---|
| 0.91 – 1.00 | Óptima | Agua sin restricciones de uso |
| 0.71 – 0.90 | Aceptable | Uso condicionado con monitoreo |
| 0.51 – 0.70 | Regular | Requiere tratamiento para uso humano |
| 0.26 – 0.50 | Deficiente | Alto riesgo sanitario |
| 0.00 – 0.25 | Muy deficiente | No apta para ningún uso |

> **ADR-008:** Usar `exceedance_report(df["od"], variable="od")` para la tabla de excedencias normativa automática. Usar `compliance_report()` para generar el HTML completo de cumplimiento del PORH.

In [ ]:
rep = exceedance_report(df["od"], variable="od")
if rep.empty:
    print("Sin normas colombianas registradas para 'od'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento

In [ ]:
df_clean = impute(df.copy(), cols=["od"], method="linear")
print(f"Faltantes antes: {df["od"].isna().sum()} | "
      f"después: {df_clean["od"].isna().sum()}")

## 7. Modelos predictivos — Métricas primarias NSE, KGE y PBIAS

### ¿Por qué NSE y KGE en vez de solo RMSE?

Para variables de calidad del agua (OD, DBO₅), el RMSE tiene el mismo problema que en caudales: no discrimina entre un modelo que acierta en la media y uno que captura la dinámica real. Para el PORH, lo crítico es predecir correctamente los **períodos de crisis** (OD mínimo en estiaje, DBO₅ máxima en eventos de vertimiento).

### NSE — Nash-Sutcliffe Efficiency

| NSE | Evaluación |
|---|---|
| > 0.75 | Bueno — modelo confiable para el PORH |
| 0.65 – 0.75 | Satisfactorio |
| 0.50 – 0.65 | Aceptable solo para análisis exploratorio |
| < 0.50 | No confiable para decisiones normativas |

### KGE — Kling-Gupta Efficiency

Para calidad del agua, los tres componentes del KGE tienen interpretación específica:
- **r (correlación):** ¿El modelo captura cuándo mejora o empeora la calidad?
- **α (variabilidad):** ¿El modelo replica la magnitud de los picos de DBO₅ o las caídas de OD?
- **β (sesgo):** ¿El modelo sobreestima o subestima sistemáticamente el OD medio?

### PBIAS — Percent Bias

```
PBIAS = 100 × Σ(Qobs - Qsim) / Σ(Qobs)
```

El PBIAS es especialmente útil para calidad del agua porque detecta sesgos sistemáticos:
- PBIAS > 0: el modelo subestima sistemáticamente (predice OD más alto del real — peligroso para el PORH).
- PBIAS < 0: el modelo sobreestima (predice OD más bajo — conservador pero costoso en decisiones de restricción).
- PBIAS óptimo: entre -10% y +10%.

### ¿Qué modelo elegir para el PORH?

- **SARIMA:** Buen punto de partida. Captura estacionalidad y autocorrelación de la calidad del agua.
- **SARIMAX + caudal:** Mejora si la calidad está fuertemente correlacionada con el régimen hidrológico (efecto de dilución). Incluir el caudal como exógena.
- **XGBoost / Random Forest:** Útil cuando hay múltiples variables de calidad correlacionadas (OD, DBO₅, temperatura). Captura no-linealidades del proceso de autodepuración.

> **ADR-003:** Para OD (variable positiva acotada entre 0 y ~14 mg/L), usar MAE + NSE + KGE como métricas primarias. No usar RMSLE. El RMSE puede ser útil como referencia secundaria.

In [ ]:
ts = df_clean.set_index("fecha")["od"]

models = {
    "SARIMA":       get_model("sarima", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "SARIMAX":      get_model("sarimax", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
    "RandomForest": get_model("random_forest", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, gap=12, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 8. Conclusiones metodológicas

### Síntesis del ciclo estadístico para recurso hídrico (PORH)

Este notebook implementó el ciclo estadístico completo para la línea `recurso_hidrico`:

1. **Validación de dominio** (`validate()`): verificación de rangos físicos de OD, DBO₅, pH, SST.
2. **EDA**: caracterización de la calidad del agua por variable y período hidrológico.
3. **ENSO con lag**: incorporación de la fase climática como modulador de la calidad.
4. **Estacionariedad** (ADF + KPSS): verificación pre-ARIMA (ADR-004).
5. **Tendencia** (Mann-Kendall): detección de deterioro progresivo o mejora de la calidad.
6. **Excedencias** (`exceedance_report()`): evaluación contra Res. 2115/2007 y Res. 631/2015.
7. **Modelos**: comparación con métricas NSE, KGE, PBIAS — no solo RMSE.

### Decisiones metodológicas clave

| Decisión | Regla aplicada | ADR |
|---|---|---|
| No eliminar outliers de DBO₅ o SST | Picos = eventos de vertimiento o lluvia real | ADR-002 |
| ADF + KPSS obligatorios | Estacionariedad pre-ARIMA | ADR-004 |
| Umbrales en config.py | No hardcodear 4 mg/L o 90 mg/L en notebook | ADR-005 |
| ENSO con lag por línea | Efecto ENSO sobre calidad vía dilución del caudal | ADR-007 |
| NSE y KGE primarios | Métricas de calidad del modelo para PORH | ADR-003 |

### Hallazgos de referencia (datos sintéticos)

Al trabajar con datos reales, documentar:
- Tendencia Mann-Kendall en OD (¿deterioro progresivo?).
- Porcentaje de meses con OD < 4 mg/L (¿estacional o permanente?).
- ICA mensual promedio y categoría IDEAM.
- Mejor modelo por NSE y KGE.
- Registrar decisiones en `docs/decisiones.md`.

### Normativa y referencias
- `docs/fuentes/recurso_hidrico.md` — Ficha técnica completa PORH
- Resolución 631 de 2015 (MADS) — Vertimientos puntuales
- Resolución 2115 de 2007 (MADS) — Agua potable
- Resolución 0751 de 2018 (MADS) — Guía técnica PORH
- IDEAM — Protocolos de monitoreo de calidad del agua

## 9. Cómo adaptar a datos reales

Esta sección es la guía práctica para quien quiera ejecutar el notebook con datos reales de un PORH o campaña de monitoreo de calidad del agua.

### Paso 1 — Obtener los datos

**Fuentes recomendadas según el tipo de análisis:**

| Tipo de dato | Fuente | Cómo acceder |
|---|---|---|
| Calidad del agua (OD, DBO₅, etc.) | Campañas de monitoreo de CARs | Solicitud directa a la CAR o IDEAM |
| Series de calidad nacionales | IDEAM — DHIME | dhime.ideam.gov.co |
| Resultados de laboratorio | Laboratorios acreditados por IDEAM | Formato electrónico del PSMV |
| Concesiones y vertimientos | SIRH — RURH | sirh.ideam.gov.co |
| Caudales para dilución | IDEAM — DHIME | dhime.ideam.gov.co |

### Paso 2 — Preparar el archivo CSV

Para análisis multivariable de calidad del agua:

```
fecha,od,dbo5,dqo,ph,sst,caudal,estacion
2018-01-15,6.2,12.5,45.0,7.2,25.0,1.8,RIO_BOGOTA_PTE_CALLE80
2018-02-15,5.8,14.2,52.0,7.0,30.0,1.5,RIO_BOGOTA_PTE_CALLE80
...
```

- `fecha`: fecha de la campaña de muestreo.
- Todas las variables de calidad en sus unidades estándar (mg/L para OD, DBO₅, DQO, SST).
- `estacion`: código del punto de monitoreo — permite filtrar por tramo de río.
- Guardar en `data/raw/recurso_hidrico.csv`.

### Paso 3 — Activar la carga real

```python
df = load_csv("data/raw/recurso_hidrico.csv", date_col="fecha")
```

Si los monitoreos son trimestrales y no mensuales, el análisis de estacionariedad ADF/KPSS sigue siendo válido, pero los modelos SARIMA deben ajustarse con `seasonal_order=(P,D,Q,4)` para estacionalidad anual en series trimestrales.

### Paso 4 — Analizar variable por variable

El notebook base usa solo `od`. Para producción, iterar sobre todas las variables:

```python
for variable in ["od", "dbo5", "dqo", "ph", "sst"]:
    ts = df_clean.set_index("fecha")[variable].dropna()
    stationarity_report(ts)
    mk = mann_kendall(ts)
    rep = exceedance_report(df[variable], variable=variable)
    print(f"\n=== {variable.upper()} ===")
    print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f}")
    display(rep)
```

### Paso 5 — Generar el reporte de cumplimiento normativo

```python
from estadistica_ambiental.reporting.compliance_report import compliance_report

compliance_report(
    df,
    variables=["od", "dbo5", "dqo", "ph", "sst"],
    linea_tematica="recurso_hidrico",
    output="data/output/reports/cumplimiento_porh.html"
)
```

### Checklist de verificación antes de entregar

- [ ] Todas las variables validadas con `validate()` (ADR-006).
- [ ] ADF + KPSS ejecutados para cada variable (ADR-004).
- [ ] ENSO incorporado con `enso_lagged()` (ADR-007).
- [ ] Excedencias calculadas contra Res. 2115/2007 y Res. 631/2015 (ADR-008).
- [ ] ICA calculado y categorizado según `config.ICA_CATEGORIES`.
- [ ] Modelo seleccionado con NSE > 0.65 (ADR-003).
- [ ] Reporte HTML generado con `compliance_report()`.
- [ ] Decisiones documentadas en `docs/decisiones.md`.